In [ ]:
import pandas as pd
import numpy as np

# ==========================
# LOAD FILES
# ==========================

file1="cleaned_LARPDatabase.xlsx"
file2="cleaned_LARPDatabase_2.xlsx"

df1=pd.read_excel(
    file1,
    dtype=str
)

df2=pd.read_excel(
    file2,
    dtype=str
)

print("File 1 Shape:",df1.shape)
print("File 2 Shape:",df2.shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

def clean_columns(df):

    df.columns=(

        df.columns
        .astype(str)
        .str.strip()
        .str.replace("\n"," ",regex=False)
        .str.replace(r"\s+","_",regex=True)

    )

    return df

df1=clean_columns(df1)
df2=clean_columns(df2)

# ==========================
# STANDARDIZE COLUMN NAMES
# ==========================

mapping={

    "Category":"CATEGORY",
    "CATEGORY":"CATEGORY",

    "Status":"STATUS",
    "STATUS":"STATUS"

}

df1=df1.rename(
    columns=mapping
)

df2=df2.rename(
    columns=mapping
)

# ==========================
# REMOVE DUPLICATE COLUMNS
# ==========================

df1=df1.loc[
    :,
    ~df1.columns.duplicated()
]

df2=df2.loc[
    :,
    ~df2.columns.duplicated()
]

# ==========================
# MERGE
# ==========================

merged=pd.concat(
    [
        df1,
        df2
    ],
    ignore_index=True,
    sort=False
)

print("\nMerged Shape Before Duplicate Removal:")
print(merged.shape)

# ==========================
# REMOVE DUPLICATE ROWS
# ==========================

duplicates=merged.duplicated().sum()

merged=merged.drop_duplicates()

print("\nDuplicate Rows Removed:")
print(duplicates)

# ==========================
# RESET SI_No
# ==========================

existing=[]

for col in merged.columns:

    c=(
        str(col)
        .lower()
        .replace("_","")
        .replace(" ","")
    )

    if c in [
        "sino",
        "slno",
        "serialno"
    ]:

        existing.append(col)

merged=merged.drop(
    columns=existing,
    errors="ignore"
)

merged=merged.reset_index(
    drop=True
)

merged.insert(
    0,
    "SI_No",
    np.arange(
        1,
        len(merged)+1
    )
)

print("\nSI_No reordered")

# ==========================
# MISSING SUMMARY
# ==========================

missing=(

    merged
    .replace("",np.nan)
    .isna()
    .sum()

)

missing=missing[
    missing>0
]

print("\nMissing Values Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(merged.shape)

print("\nFinal Columns:")
print(list(merged.columns))

# ==========================
# EXPORT
# ==========================

output="merged_LARP_finals.xlsx"

with pd.ExcelWriter(
    output,
    engine="openpyxl"
) as writer:

    merged.to_excel(
        writer,
        index=False,
        sheet_name="Merged"
    )

print("\nSaved:",output)

File 1 Shape: (61073, 22)
File 2 Shape: (61050, 22)

Merged Shape Before Duplicate Removal:
(122123, 22)

Duplicate Rows Removed:
3140

SI_No reordered

Missing Values Summary:
MANAGEMENT                  1879
VESSEL                      1879
AREA_OF_CONCERN            40986
DESCRIPTION                   34
IMMEDIATE_ACTION_TAKEN     15164
WORK_EQUIPMENT            106232
LARP_PROCEDURE             67169
LARP_SAFETY_DEVICE        106013
WORK_ENVIRONMENT          110468
HOUSE_KEEPING             106412
PERMIT_TO_WORK            117697
LARP_PPE                  103012
WORK_POSITION             104124
REPORTED_BY                16162
CLOSURE_REMARK            107795
CLOSURE_DATE              107783
CLOSED_BY                 107785
dtype: int64

Final Shape:
(118983, 22)

Final Columns:
['SI_No', 'MANAGEMENT', 'VESSEL', 'REF_NO', 'OCCURRENCE_DATE', 'STATUS', 'AREA_OF_CONCERN', 'DESCRIPTION', 'IMMEDIATE_ACTION_TAKEN', 'CATEGORY', 'WORK_EQUIPMENT', 'LARP_PROCEDURE', 'LARP_SAFETY_DEVICE', 'WO